<a href="https://colab.research.google.com/github/Nghia-Trinh/Long-Short-Equity-Spring-26/blob/main/granite_timeseries_patchtsmixer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U transformers accelerate sentencepiece python-docx pypdf cvxpy pandas numpy matplotlib openpyxl

## Local Inference on GPU
Model page: https://huggingface.co/ibm-granite/granite-timeseries-patchtsmixer

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/ibm-granite/granite-timeseries-patchtsmixer)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Optional: load IBM Granite from Hugging Face for thesis scoring in Colab
from transformers import pipeline

MODEL_ID = "ibm-granite/granite-3.3-2b-instruct"

try:
    granite_pipe = pipeline(
        task="text-generation",
        model=MODEL_ID,
        tokenizer=MODEL_ID,
        trust_remote_code=True,
    )
    print("Loaded:", MODEL_ID)
except Exception as exc:
    granite_pipe = None
    print("Model load failed (will use lexical fallback in repo code):", exc)

In [ ]:
# Clone or update repository in Colab workspace
import os

REPO_URL = "https://github.com/Nghia-Trinh/Long-Short-Equity-Spring-26.git"
REPO_DIR = "/content/Long-Short-Equity-Spring-26"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!pip -q install -r requirements.txt
print("Ready in", REPO_DIR)

In [ ]:
# Configure thesis overlay for this Colab session
import json
from pathlib import Path

cfg_path = Path("config.json")
config = json.loads(cfg_path.read_text())

config.update({
    "llm_overlay_enabled": True,
    "llm_use_hf_model": granite_pipe is not None,
    "llm_model_id": MODEL_ID,
    "llm_thesis_dir": "Investment Theses",
    # Optional: provide your own file in repo and set path here
    "llm_member_selection_file": "Investment Theses/member_selections.sample.json",
})

cfg_path.write_text(json.dumps(config, indent=4))
print("Updated config with LLM overlay settings")

In [ ]:
# Run thesis-aware backtest and generate season diagnostics
!python3 backtest.py --enable-llm-overlay --llm-thesis-dir "Investment Theses" --llm-member-selections "Investment Theses/member_selections.sample.json"

In [ ]:
# Inspect outputs for member-selection adjustments and earnings-season results
import pandas as pd

weights = pd.read_csv("outputs/portfolio_weights.csv")
pnl = pd.read_csv("outputs/pnl.csv")
seasons = pd.read_csv("outputs/earnings_season_metrics.csv")

print("Weights shape:", weights.shape)
print("PnL rows:", len(pnl))
print("Season rows:", len(seasons))

display(seasons.tail(8))

diag_path = "outputs/llm_overlay_diagnostics.csv"
try:
    diag = pd.read_csv(diag_path)
    display(diag.tail(8))
except Exception:
    print("No LLM diagnostics file found (likely no thesis/member ticker matches).")

In [ ]:
# Optional: run only a prior earnings season window for focused replay
import json
from pathlib import Path
import pandas as pd

seasons = pd.read_csv("outputs/earnings_season_metrics.csv")
if len(seasons) > 0:
    target = seasons.iloc[-1]
    start = str(pd.to_datetime(target["start_date"]).date())
    end = str(pd.to_datetime(target["end_date"]).date())

    cfg_path = Path("config.json")
    cfg = json.loads(cfg_path.read_text())
    cfg["start_date"] = start
    cfg["end_date"] = end
    cfg_path.write_text(json.dumps(cfg, indent=4))

    print(f"Replaying season {target['season']} from {start} to {end}")
    !python3 backtest.py --enable-llm-overlay --llm-thesis-dir "Investment Theses" --llm-member-selections "Investment Theses/member_selections.sample.json"
else:
    print("No season rows found. Run full backtest cell first.")